# Call Brief Generator**Objective** — Build a workflow that:1. Gathers structured company information (firmographics, recent news, tech stack)2. Ingests account history and CRM notes (past meetings, deal stage, key contacts)3. Accepts upcoming meeting context (agenda, attendees, goals)4. Synthesises a one-page **call brief** with prioritised **talking points**5. Adapts tone and depth to meeting type (discovery, QBR, renewal, executive)This notebook demonstrates the full pipeline with realistic synthetic data,evaluates brief quality, and includes production-ready patterns.

## Why Automated Call Briefs MatterSales and customer-success teams spend 30-60 minutes manually preparingfor each important call — pulling CRM data, reading past notes, scanningLinkedIn, and drafting an agenda. An AI-generated call brief:| Benefit | Impact ||---|---|| **Time savings** | 5-min read replaces 30-min research || **Consistency** | Every rep walks in equally prepared || **Context preservation** | Knowledge doesn't leave when reps change || **Personalisation** | Talking points reference the prospect's own words || **Confidence** | Reps handle tough questions with pre-loaded answers |### Pipeline Architecture```Company Profile ──┐                   │Account History ───┼──► Context Assembler ──► Brief Generator ──► Call Brief                   │                                │Meeting Context ───┘                    Talking-Point Ranker ──► Ranked Points```

In [ ]:
from __future__ import annotationsimport mathimport reimport textwrapfrom collections import Counter, defaultdictfrom dataclasses import dataclass, fieldfrom datetime import datetime, timedeltafrom typing import Optionalimport numpy as npimport pandas as pdimport plotly.express as pximport plotly.graph_objects as gofrom plotly.subplots import make_subplotsprint('Libraries loaded.')

## 1 · Data ModelsWe model three input layers that mirror what a CRM + enrichment platform provides:- **CompanyProfile** — firmographics, news, tech stack, competitors- **AccountHistory** — CRM notes, past meetings, deal stage, contacts- **MeetingContext** — upcoming meeting details, attendees, goals

In [ ]:
@dataclassclass CompanyProfile:    name: str    industry: str    hq_location: str    employee_count: int    annual_revenue: str    founded: int    website: str    tech_stack: list[str]    competitors: list[str]    recent_news: list[str]    key_initiatives: list[str]    def summary_text(self) -> str:        news_str = '; '.join(self.recent_news[:3])        return (            f'{self.name} ({self.industry}) | {self.hq_location} | '            f'{self.employee_count} employees | Rev: {self.annual_revenue} | '            f'Tech: {", ".join(self.tech_stack[:5])} | News: {news_str}'        )@dataclassclass AccountNote:    date: str    author: str    note_type: str  # meeting, email, internal, call    content: str    sentiment: str = 'neutral'  # positive, neutral, negative, concern@dataclassclass Contact:    name: str    title: str    role: str  # champion, decision_maker, influencer, blocker, end_user    email: str    notes: str = ''@dataclassclass AccountHistory:    account_id: str    deal_stage: str    deal_value: str    contract_end: str    health_score: float  # 0-100    contacts: list[Contact]    notes: list[AccountNote]    open_issues: list[str]    product_usage: dict  # feature -> adoption %    def timeline_text(self) -> str:        sorted_notes = sorted(self.notes, key=lambda n: n.date, reverse=True)        return '\n'.join(            f'[{n.date}] ({n.note_type}) {n.content}'            for n in sorted_notes[:8]        )@dataclassclass MeetingContext:    meeting_type: str  # discovery, demo, qbr, renewal, executive, support    scheduled_date: str    duration_min: int    our_attendees: list[str]    their_attendees: list[str]    agenda_items: list[str]    goals: list[str]    special_notes: str = ''print('Data models defined.')

## 2 · Synthetic Account DataWe create 6 realistic accounts spanning different industries, deal stages,and meeting types. Each account has a full company profile, CRM historywith timestamped notes, and an upcoming meeting with specific goals.

In [ ]:
BASE = datetime(2026, 4, 12)# ── Account 1: Enterprise SaaS renewal ──────────────────────────────────company1 = CompanyProfile(    name='Meridian Financial Services',    industry='Financial Services',    hq_location='New York, NY',    employee_count=4200,    annual_revenue='$820M',    founded=1998,    website='meridianfs.com',    tech_stack=['Salesforce', 'Snowflake', 'AWS', 'Tableau', 'ServiceNow'],    competitors=['Goldman Tech Solutions', 'FinServ Digital', 'Apex Analytics'],    recent_news=[        'Q1 2026: Announced 15% revenue growth and expansion into Asia-Pacific',        'Mar 2026: Hired new CTO from Stripe, signaling tech modernization push',        'Feb 2026: Regulatory compliance audit flagged data lineage gaps',    ],    key_initiatives=['Data governance overhaul', 'Cloud migration Phase 2', 'AI/ML for fraud detection'],)account1 = AccountHistory(    account_id='ACCT-001',    deal_stage='renewal',    deal_value='$340K ARR',    contract_end='2026-06-30',    health_score=72.0,    contacts=[        Contact('David Chen', 'VP of Data Engineering', 'champion', 'd.chen@meridianfs.com',                'Strong advocate; drove initial purchase. Concerned about pricing.'),        Contact('Sarah Kim', 'CTO', 'decision_maker', 's.kim@meridianfs.com',                'New hire from Stripe (Mar 2026). Unknown disposition toward our platform.'),        Contact('Mark Rodriguez', 'Director of Compliance', 'influencer', 'm.rodriguez@meridianfs.com',                'Needs data lineage features. Positive on recent improvements.'),        Contact('Janet Liu', 'CFO', 'decision_maker', 'j.liu@meridianfs.com',                'Budget-conscious. Previously pushed back on 10% price increase.'),    ],    notes=[        AccountNote('2026-04-08', 'Alex Martinez', 'meeting',                    'QBR with David Chen. Usage up 22% YoY. He mentioned CTO wants to '                    'evaluate alternatives before renewal. Need to get in front of Sarah Kim.',                    'concern'),        AccountNote('2026-03-25', 'Alex Martinez', 'email',                    'David confirmed compliance team is happy with new lineage features. '                    'Asked about multi-region support timeline.',                    'positive'),        AccountNote('2026-03-15', 'Priya Sharma', 'internal',                    'Support ticket #4892: Data pipeline latency spikes during peak hours. '                    'Engineering working on fix, ETA April.',                    'negative'),        AccountNote('2026-02-20', 'Alex Martinez', 'call',                    'Intro call with Mark Rodriguez (Compliance). Very interested in '                    'audit trail and data classification features. Wants a demo.',                    'positive'),        AccountNote('2026-01-10', 'Alex Martinez', 'meeting',                    'Annual planning. David shared 2026 priorities: cloud migration, '                    'data governance, cost optimisation. Budget is flat YoY.',                    'neutral'),    ],    open_issues=['Pipeline latency during peak (ticket #4892)', 'Multi-region support request'],    product_usage={'Data Pipelines': 89, 'Data Catalog': 67, 'Lineage': 45, 'Governance': 38, 'API Access': 92},)meeting1 = MeetingContext(    meeting_type='renewal',    scheduled_date='2026-04-15',    duration_min=60,    our_attendees=['Alex Martinez (AE)', 'Priya Sharma (CSM)', 'Tom Harris (SE)'],    their_attendees=['David Chen (VP Data Eng)', 'Sarah Kim (CTO)', 'Janet Liu (CFO)'],    agenda_items=[        'Renewal terms and pricing discussion',        'Product roadmap alignment with 2026 priorities',        'Open issue resolution: pipeline latency',        'Multi-region support timeline',    ],    goals=[        'Secure verbal commitment to renew at current or increased rate',        'Build relationship with new CTO Sarah Kim',        'Address pipeline latency concern with engineering update',        'Position data governance roadmap to align with compliance needs',    ],    special_notes='First meeting with CTO Sarah Kim. She may compare us to Stripe internal tooling.',)# ── Account 2: Mid-market discovery ─────────────────────────────────────company2 = CompanyProfile(    name='GreenLoop Logistics',    industry='Supply Chain / Logistics',    hq_location='Austin, TX',    employee_count=650,    annual_revenue='$95M',    founded=2015,    website='greenloop.io',    tech_stack=['HubSpot', 'PostgreSQL', 'GCP', 'Looker', 'dbt'],    competitors=['FleetOps', 'RouteWise', 'LogiTrack'],    recent_news=[        'Mar 2026: Series C ($45M) led by Sequoia to expand into European markets',        'Feb 2026: Launched same-day delivery service for enterprise clients',        'Jan 2026: Named in Forbes "Next Billion-Dollar Startups" list',    ],    key_initiatives=['European expansion', 'Real-time fleet analytics', 'Carbon footprint tracking'],)account2 = AccountHistory(    account_id='ACCT-002',    deal_stage='discovery',    deal_value='$85K ARR (est.)',    contract_end='N/A',    health_score=0.0,    contacts=[        Contact('Ryan Park', 'Head of Analytics', 'champion', 'r.park@greenloop.io',                'Reached out via website demo request. Frustrated with current Looker setup.'),        Contact('Emma Walsh', 'VP Engineering', 'decision_maker', 'e.walsh@greenloop.io',                'Runs platform team. Final technical sign-off needed.'),    ],    notes=[        AccountNote('2026-04-10', 'Lisa Nguyen', 'email',                    'Inbound from Ryan Park. Needs real-time dashboards for fleet tracking. '                    'Current Looker setup is too slow for operational decisions.',                    'positive'),        AccountNote('2026-04-11', 'Lisa Nguyen', 'internal',                    'Researched GreenLoop: Series C, growing fast, GCP stack. '                    'Good fit for our real-time analytics module. Competitor FleetOps uses Datadog.',                    'neutral'),    ],    open_issues=[],    product_usage={},)meeting2 = MeetingContext(    meeting_type='discovery',    scheduled_date='2026-04-14',    duration_min=30,    our_attendees=['Lisa Nguyen (AE)', 'James Carter (SE)'],    their_attendees=['Ryan Park (Head of Analytics)'],    agenda_items=[        'Understand current analytics stack and pain points',        'Discuss real-time dashboard requirements',        'Explore data volume and integration needs',    ],    goals=[        'Qualify opportunity: confirm budget, authority, need, timeline',        'Identify 2-3 specific pain points we can solve',        'Secure intro to VP Engineering for technical evaluation',    ],)# ── Account 3: Executive QBR ────────────────────────────────────────────company3 = CompanyProfile(    name='NovaCare Health Systems',    industry='Healthcare / Health Tech',    hq_location='Boston, MA',    employee_count=8500,    annual_revenue='$2.1B',    founded=1985,    website='novacare.com',    tech_stack=['Epic', 'Azure', 'Power BI', 'Databricks', 'Kafka'],    competitors=['CarePoint Digital', 'MedFlow Analytics', 'HealthBridge'],    recent_news=[        'Q1 2026: Launched AI-powered patient triage system across 12 hospitals',        'Mar 2026: HIPAA audit — all systems compliant, but data integration flagged',        'Feb 2026: Acquired MedInsight Analytics for $180M to bolster data capabilities',    ],    key_initiatives=['Clinical data unification', 'Patient experience analytics', 'HIPAA-compliant AI deployment'],)account3 = AccountHistory(    account_id='ACCT-003',    deal_stage='expansion',    deal_value='$720K ARR (current) + $280K expansion',    contract_end='2027-01-15',    health_score=88.0,    contacts=[        Contact('Dr. Rachel Okafor', 'CMIO', 'decision_maker', 'r.okafor@novacare.com',                'Executive sponsor. Personally championed initial rollout. Presenting to board Q2.'),        Contact('Kevin Wu', 'Director of Data Platforms', 'champion', 'k.wu@novacare.com',                'Day-to-day owner. Very technical. Loves our Kafka integration.'),        Contact('Linda Torres', 'VP of IT', 'influencer', 'l.torres@novacare.com',                'Concerned about vendor consolidation. Prefers Azure-native tools.'),    ],    notes=[        AccountNote('2026-04-05', 'Sam Patel', 'meeting',                    'QBR prep with Kevin Wu. Cluster processing 2.4B events/day, up 40% from Q4. '                    'Wants to expand to 3 more hospital systems. Budget approved in principle.',                    'positive'),        AccountNote('2026-03-20', 'Sam Patel', 'email',                    'Dr. Okafor wants ROI data for board presentation. Need to prepare '                    'cost-savings and patient-outcome improvement metrics by April 10.',                    'neutral'),        AccountNote('2026-03-10', 'Sam Patel', 'internal',                    'Linda Torres asked about Azure-native alternatives during IT review. '                    'We need to emphasize our Azure Marketplace listing and native integrations.',                    'concern'),        AccountNote('2026-02-28', 'Sam Patel', 'call',                    'Kevin confirmed MedInsight acquisition means new data sources to integrate. '                    'Our platform is the top candidate for unified pipeline.',                    'positive'),    ],    open_issues=['ROI metrics for board deck needed by Apr 10', 'Azure-native positioning for Linda Torres'],    product_usage={'Streaming Ingest': 95, 'Batch Processing': 88, 'SQL Analytics': 76,                   'ML Workbench': 52, 'HIPAA Audit Logs': 91},)meeting3 = MeetingContext(    meeting_type='executive',    scheduled_date='2026-04-15',    duration_min=45,    our_attendees=['Sam Patel (AE)', 'VP of Sales (Jessica Huang)', 'CTO (Michael Brooks)'],    their_attendees=['Dr. Rachel Okafor (CMIO)', 'Kevin Wu (Dir Data Platforms)', 'Linda Torres (VP IT)'],    agenda_items=[        'Expansion proposal: 3 additional hospital systems',        'ROI review for board presentation',        'Roadmap alignment: HIPAA-compliant AI features',        'MedInsight data integration plan',    ],    goals=[        'Get Dr. Okafor approval on expansion deal',        'Provide board-ready ROI metrics',        'Neutralise Linda Torres vendor-consolidation concern',        'Position as the integration backbone for MedInsight data',    ],    special_notes='Our VP of Sales and CTO are attending. This is a strategic account — treat as executive briefing.',)# ── Account 4: Support escalation ───────────────────────────────────────company4 = CompanyProfile(    name='Velocity Commerce',    industry='E-Commerce / Retail Tech',    hq_location='Seattle, WA',    employee_count=1200,    annual_revenue='$210M',    founded=2012,    website='velocitycommerce.com',    tech_stack=['Shopify Plus', 'AWS', 'Redshift', 'Segment', 'Braze'],    competitors=['CommerceHub', 'ShopFlow', 'RetailIQ'],    recent_news=[        'Apr 2026: Website outage during flash sale caused $2M estimated lost revenue',        'Mar 2026: Launched personalization engine powered by in-house ML team',        'Feb 2026: Expanded into B2B wholesale channel',    ],    key_initiatives=['Real-time inventory sync', 'Personalisation at scale', 'B2B channel growth'],)account4 = AccountHistory(    account_id='ACCT-004',    deal_stage='at_risk',    deal_value='$165K ARR',    contract_end='2026-09-30',    health_score=41.0,    contacts=[        Contact('Tina Pham', 'VP of Engineering', 'decision_maker', 't.pham@velocitycommerce.com',                'Frustrated after flash-sale outage. Blames our data sync for inventory mismatch.'),        Contact('Derek Foster', 'Data Engineer', 'end_user', 'd.foster@velocitycommerce.com',                'Technically competent. Has been helpful in debugging. Wants better docs.'),    ],    notes=[        AccountNote('2026-04-09', 'Carlos Mendez', 'call',                    'Escalation call with Tina Pham. She is considering switching to '                    'CommerceHub if latency issues are not resolved by end of April.',                    'negative'),        AccountNote('2026-04-07', 'Carlos Mendez', 'internal',                    'Engineering root cause: our Redshift connector dropped 3% of events '                    'during peak load. Fix deployed in v4.2.1, needs customer validation.',                    'neutral'),        AccountNote('2026-04-03', 'Carlos Mendez', 'email',                    'Derek Foster confirmed he can run validation tests if we provide '                    'a testing playbook. Positive attitude despite the issues.',                    'positive'),    ],    open_issues=['Redshift connector event loss (fixed in v4.2.1, awaiting validation)',                 'Documentation gaps for real-time sync module',                 'Flash-sale outage post-mortem requested by VP Eng'],    product_usage={'Real-time Sync': 78, 'Batch ETL': 91, 'Event Streaming': 65, 'API Gateway': 88},)meeting4 = MeetingContext(    meeting_type='support',    scheduled_date='2026-04-14',    duration_min=45,    our_attendees=['Carlos Mendez (CSM)', 'Anita Rao (Engineering Lead)'],    their_attendees=['Tina Pham (VP Engineering)', 'Derek Foster (Data Engineer)'],    agenda_items=[        'Post-mortem: flash-sale outage root cause and fix',        'Validation plan for Redshift connector v4.2.1',        'Documentation improvement roadmap',        'Rebuild trust and retention path',    ],    goals=[        'Present clear root cause and permanent fix for event loss',        'Get Tina to agree to validation test before deciding on vendor switch',        'Rebuild confidence by showing proactive engineering response',        'Secure 90-day retention window to prove fix effectiveness',    ],    special_notes='HIGH RISK: Account may churn if this meeting goes poorly. Tina is evaluating CommerceHub.',)# ── Account 5: Demo/technical evaluation ────────────────────────────────company5 = CompanyProfile(    name='Pinnacle Education',    industry='EdTech',    hq_location='San Francisco, CA',    employee_count=350,    annual_revenue='$42M',    founded=2018,    website='pinnacleedu.com',    tech_stack=['React', 'Firebase', 'BigQuery', 'Airflow', 'Metabase'],    competitors=['Coursera for Business', 'Skillsoft', 'Degreed'],    recent_news=[        'Mar 2026: Launched enterprise LMS with AI-powered content recommendations',        'Feb 2026: Partnership with 50 universities for degree program integration',    ],    key_initiatives=['Learner analytics platform', 'Content recommendation engine', 'Enterprise SSO rollout'],)account5 = AccountHistory(    account_id='ACCT-005',    deal_stage='evaluation',    deal_value='$55K ARR (est.)',    contract_end='N/A',    health_score=0.0,    contacts=[        Contact('Aisha Johnson', 'Head of Data', 'champion', 'a.johnson@pinnacleedu.com',                'Technically strong. Evaluating us vs Databricks for learner analytics.'),        Contact('Chris Lee', 'CTO', 'decision_maker', 'c.lee@pinnacleedu.com',                'Budget owner. Prefers open-source-first approach.'),    ],    notes=[        AccountNote('2026-04-09', 'Lisa Nguyen', 'meeting',                    'First call with Aisha. She needs to process 500M learner events/month. '                    'Current Airflow+BigQuery setup is hitting scale limits. Wants a demo of '                    'our streaming ingest and real-time dashboard.',                    'positive'),    ],    open_issues=[],    product_usage={},)meeting5 = MeetingContext(    meeting_type='demo',    scheduled_date='2026-04-16',    duration_min=45,    our_attendees=['Lisa Nguyen (AE)', 'James Carter (SE)'],    their_attendees=['Aisha Johnson (Head of Data)', 'Chris Lee (CTO)'],    agenda_items=[        'Live demo: streaming ingest for learner events',        'Real-time dashboard walkthrough',        'BigQuery integration and migration path',        'Pricing and open-source compatibility',    ],    goals=[        'Demonstrate technical superiority over Databricks for this use case',        'Show BigQuery-native integration to ease migration concerns',        'Address CTO open-source preference with our open API approach',        'Secure POC commitment with success criteria',    ],)# ── Account 6: QBR ──────────────────────────────────────────────────────company6 = CompanyProfile(    name='Atlas Manufacturing',    industry='Manufacturing / Industrial',    hq_location='Detroit, MI',    employee_count=3100,    annual_revenue='$580M',    founded=1972,    website='atlasmfg.com',    tech_stack=['SAP', 'Azure', 'Power BI', 'Historian', 'OSIsoft PI'],    competitors=['Siemens Digital', 'Rockwell Automation', 'PTC ThingWorx'],    recent_news=[        'Q1 2026: Invested $50M in smart factory initiative across 4 plants',        'Mar 2026: IoT sensor deployment completed on production line A',    ],    key_initiatives=['Smart factory / Industry 4.0', 'Predictive maintenance', 'Supply chain visibility'],)account6 = AccountHistory(    account_id='ACCT-006',    deal_stage='active',    deal_value='$195K ARR',    contract_end='2027-03-31',    health_score=81.0,    contacts=[        Contact('Bob Kowalski', 'VP Operations', 'decision_maker', 'b.kowalski@atlasmfg.com',                'Practical, results-driven. Wants to see downtime-reduction metrics.'),        Contact('Mei Zhang', 'IoT Platform Lead', 'champion', 'm.zhang@atlasmfg.com',                'Technical champion. Built the initial integration. Very engaged.'),    ],    notes=[        AccountNote('2026-04-01', 'Sam Patel', 'meeting',                    'QBR prep. Mei shared that predictive maintenance model reduced '                    'unplanned downtime by 18% in Q1. Bob wants to extend to Line B and C.',                    'positive'),        AccountNote('2026-03-15', 'Sam Patel', 'email',                    'Bob asked for a comparison report vs Siemens Digital for Lines B & C. '                    'Needs justification for expanding our contract vs a competitor pilot.',                    'concern'),    ],    open_issues=['Competitive comparison vs Siemens requested', 'Line B/C sensor data format TBD'],    product_usage={'IoT Ingest': 88, 'Time-Series DB': 82, 'Predictive Models': 71, 'Dashboards': 64},)meeting6 = MeetingContext(    meeting_type='qbr',    scheduled_date='2026-04-17',    duration_min=60,    our_attendees=['Sam Patel (AE)', 'Engineering Manager (Rita Flores)'],    their_attendees=['Bob Kowalski (VP Ops)', 'Mei Zhang (IoT Lead)'],    agenda_items=[        'Q1 results: downtime reduction and ROI metrics',        'Expansion plan for Lines B and C',        'Competitive positioning vs Siemens Digital',        'Sensor data format and integration roadmap',    ],    goals=[        'Present compelling Q1 ROI to justify expansion',        'Get Bob to commit to Lines B & C expansion',        'Pre-empt Siemens comparison with differentiation points',        'Align on sensor data format for new lines',    ],)# Bundle all accountsACCOUNTS = [    (company1, account1, meeting1),    (company2, account2, meeting2),    (company3, account3, meeting3),    (company4, account4, meeting4),    (company5, account5, meeting5),    (company6, account6, meeting6),]print(f'Accounts loaded: {len(ACCOUNTS)}')for co, acct, mtg in ACCOUNTS:    print(f'  {acct.account_id}: {co.name} | {acct.deal_stage} | {mtg.meeting_type} | '          f'health={acct.health_score}')

## 3 · Context AssemblerThe context assembler collects and structures all available informationinto a single document that the brief generator can consume. It:1. Merges company profile, account history, and meeting context2. Identifies key risks and opportunities from the notes3. Maps attendee roles and known dispositions4. Computes urgency signals (contract proximity, health score, deal stage)

In [ ]:
@dataclassclass AssembledContext:    company: CompanyProfile    account: AccountHistory    meeting: MeetingContext    risks: list[str] = field(default_factory=list)    opportunities: list[str] = field(default_factory=list)    attendee_map: dict = field(default_factory=dict)    urgency_level: str = 'normal'  # low, normal, high, critical    sentiment_trend: str = 'stable'RISK_KEYWORDS = ['churn', 'switch', 'alternative', 'competitor', 'frustrated',                 'escalation', 'outage', 'issue', 'budget', 'pricing', 'concern',                 'consolidation', 'evaluate']OPPORTUNITY_KEYWORDS = ['expand', 'growth', 'initiative', 'budget approved',                        'advocate', 'champion', 'interested', 'impressed',                        'increase', 'adoption', 'rollout']def assemble_context(company: CompanyProfile, account: AccountHistory,                     meeting: MeetingContext) -> AssembledContext:    ctx = AssembledContext(company=company, account=account, meeting=meeting)    # ── Extract risks and opportunities from notes ──    for note in account.notes:        text_lower = note.content.lower()        for kw in RISK_KEYWORDS:            if kw in text_lower:                ctx.risks.append(f'[{note.date}] {note.content[:120]}')                break        for kw in OPPORTUNITY_KEYWORDS:            if kw in text_lower:                ctx.opportunities.append(f'[{note.date}] {note.content[:120]}')                break    # ── Add open issues as risks ──    for issue in account.open_issues:        ctx.risks.append(f'[OPEN ISSUE] {issue}')    # ── Map attendees to known contacts ──    contact_lookup = {c.name.lower(): c for c in account.contacts}    for attendee in meeting.their_attendees:        name_part = attendee.split('(')[0].strip().lower()        if name_part in contact_lookup:            c = contact_lookup[name_part]            ctx.attendee_map[c.name] = {                'title': c.title, 'role': c.role, 'notes': c.notes            }    # ── Compute urgency ──    if account.deal_stage == 'at_risk' or account.health_score < 50:        ctx.urgency_level = 'critical'    elif account.deal_stage == 'renewal':        if account.contract_end != 'N/A':            days_left = (datetime.fromisoformat(account.contract_end).date()                         - BASE.date()).days            ctx.urgency_level = 'high' if days_left < 90 else 'normal'        else:            ctx.urgency_level = 'high'    elif meeting.meeting_type == 'executive':        ctx.urgency_level = 'high'    # ── Compute sentiment trend ──    sentiments = [n.sentiment for n in account.notes]    if sentiments:        recent = sentiments[:3]        neg_count = sum(1 for s in recent if s in ('negative', 'concern'))        pos_count = sum(1 for s in recent if s == 'positive')        if neg_count >= 2:            ctx.sentiment_trend = 'declining'        elif pos_count >= 2:            ctx.sentiment_trend = 'improving'    return ctx# Assemble all contextsassembled = [assemble_context(co, acct, mtg) for co, acct, mtg in ACCOUNTS]for ctx in assembled:    print(f'{ctx.account.account_id}: {ctx.company.name} | '          f'urgency={ctx.urgency_level} | sentiment={ctx.sentiment_trend} | '          f'risks={len(ctx.risks)} | opportunities={len(ctx.opportunities)}')

## 4 · Talking-Point GeneratorTalking points are ranked by relevance to the meeting type and attendees.Each point has a **category**, **priority** (must-say / should-say / nice-to-say),and optionally a **source** (which note or data point it came from).### Meeting-Type Templates| Type | Focus ||---|---|| Discovery | Pain points, qualification, next steps || Demo | Technical fit, differentiation, POC criteria || QBR | ROI metrics, usage trends, expansion || Renewal | Value delivered, pricing, risk mitigation || Executive | Strategic alignment, business outcomes, partnership || Support | Issue resolution, trust rebuilding, retention |

In [ ]:
@dataclassclass TalkingPoint:    category: str    text: str    priority: str  # must_say, should_say, nice_to_say    source: str = ''    target_attendee: str = ''MEETING_TYPE_PRIORITIES = {    'discovery': ['pain_points', 'qualification', 'next_steps', 'company_context'],    'demo': ['technical_fit', 'differentiation', 'poc_criteria', 'pricing'],    'qbr': ['roi_metrics', 'usage_trends', 'expansion', 'issues'],    'renewal': ['value_delivered', 'pricing', 'risk_mitigation', 'roadmap'],    'executive': ['strategic_alignment', 'business_outcomes', 'partnership', 'competitive'],    'support': ['issue_resolution', 'trust_rebuilding', 'retention', 'next_steps'],}def generate_talking_points(ctx: AssembledContext) -> list[TalkingPoint]:    points = []    mt = ctx.meeting.meeting_type    priorities = MEETING_TYPE_PRIORITIES.get(mt, ['company_context'])    # ── Company context points ──    for news in ctx.company.recent_news[:2]:        points.append(TalkingPoint(            category='company_context',            text=f'Reference recent development: {news}',            priority='nice_to_say',            source='company_news',        ))    for init in ctx.company.key_initiatives[:2]:        points.append(TalkingPoint(            category='strategic_alignment',            text=f'Align our value to their initiative: {init}',            priority='should_say' if mt in ('executive', 'qbr', 'renewal') else 'nice_to_say',            source='company_profile',        ))    # ── Risk-based points ──    for risk in ctx.risks:        if 'OPEN ISSUE' in risk:            cat = 'issue_resolution' if mt == 'support' else 'risk_mitigation'            points.append(TalkingPoint(                category=cat,                text=f'Address: {risk.replace("[OPEN ISSUE] ", "")}',                priority='must_say',                source='open_issues',            ))        else:            points.append(TalkingPoint(                category='risk_mitigation',                text=f'Be prepared to discuss: {risk[13:]}',                priority='should_say',                source='account_notes',            ))    # ── Opportunity-based points ──    for opp in ctx.opportunities:        points.append(TalkingPoint(            category='expansion' if mt in ('qbr', 'renewal', 'executive') else 'qualification',            text=f'Leverage: {opp[13:]}',            priority='should_say',            source='account_notes',        ))    # ── Attendee-specific points ──    for name, info in ctx.attendee_map.items():        if info['role'] == 'decision_maker':            points.append(TalkingPoint(                category='strategic_alignment',                text=f'Engage {name} ({info["title"]}): {info["notes"]}',                priority='must_say',                target_attendee=name,                source='contact_notes',            ))        elif info['role'] == 'blocker':            points.append(TalkingPoint(                category='risk_mitigation',                text=f'Address concerns of {name} ({info["title"]}): {info["notes"]}',                priority='must_say',                target_attendee=name,                source='contact_notes',            ))    # ── Meeting goals as points ──    for goal in ctx.meeting.goals:        points.append(TalkingPoint(            category='next_steps',            text=f'Drive toward: {goal}',            priority='must_say',            source='meeting_goals',        ))    # ── Usage/adoption points for existing customers ──    if ctx.account.product_usage:        low_adoption = [(f, pct) for f, pct in ctx.account.product_usage.items() if pct < 60]        high_adoption = [(f, pct) for f, pct in ctx.account.product_usage.items() if pct >= 85]        if high_adoption:            features = ', '.join(f'{f} ({p}%)' for f, p in high_adoption)            points.append(TalkingPoint(                category='value_delivered',                text=f'Highlight strong adoption: {features}',                priority='should_say',                source='product_usage',            ))        if low_adoption:            features = ', '.join(f'{f} ({p}%)' for f, p in low_adoption)            points.append(TalkingPoint(                category='usage_trends',                text=f'Explore under-adoption with enablement offer: {features}',                priority='nice_to_say' if mt == 'discovery' else 'should_say',                source='product_usage',            ))    # ── Rank by meeting-type priority order ──    priority_order = {'must_say': 0, 'should_say': 1, 'nice_to_say': 2}    cat_order = {cat: i for i, cat in enumerate(priorities)}    def sort_key(tp: TalkingPoint):        return (priority_order.get(tp.priority, 2),                cat_order.get(tp.category, 99))    points.sort(key=sort_key)    return points# Generate for all accountsall_points = {}for ctx in assembled:    pts = generate_talking_points(ctx)    all_points[ctx.account.account_id] = pts    must = sum(1 for p in pts if p.priority == 'must_say')    should = sum(1 for p in pts if p.priority == 'should_say')    nice = sum(1 for p in pts if p.priority == 'nice_to_say')    print(f'{ctx.account.account_id}: {len(pts)} talking points '          f'(must={must}, should={should}, nice={nice})')

## 5 · Call Brief GeneratorThe brief generator synthesises all assembled context and talking pointsinto a structured one-page document. The format adapts to meeting type:- **Header** — Company, date, attendees, meeting type- **Executive summary** — 2-3 sentences on account status and meeting purpose- **Key risks & opportunities** — Prioritised bullet list- **Attendee intelligence** — What we know about each person in the room- **Talking points** — Ranked by priority- **Preparation checklist** — What to review or prepare before the call

In [ ]:
@dataclassclass CallBrief:    account_id: str    company_name: str    meeting_type: str    urgency: str    header: str    executive_summary: str    risks_section: str    opportunities_section: str    attendee_intel: str    talking_points_section: str    prep_checklist: str    word_count: int = 0    def full_text(self) -> str:        sections = [            self.header,            '\n--- EXECUTIVE SUMMARY ---',            self.executive_summary,            '\n--- KEY RISKS ---',            self.risks_section,            '\n--- OPPORTUNITIES ---',            self.opportunities_section,            '\n--- ATTENDEE INTELLIGENCE ---',            self.attendee_intel,            '\n--- TALKING POINTS ---',            self.talking_points_section,            '\n--- PREPARATION CHECKLIST ---',            self.prep_checklist,        ]        return '\n'.join(sections)def generate_brief(ctx: AssembledContext,                   points: list[TalkingPoint]) -> CallBrief:    co = ctx.company    acct = ctx.account    mtg = ctx.meeting    # ── Header ──    header_lines = [        f'CALL BRIEF: {co.name}',        f'Meeting Type : {mtg.meeting_type.upper()}',        f'Date         : {mtg.scheduled_date} ({mtg.duration_min} min)',        f'Urgency      : {ctx.urgency_level.upper()}',        f'Our Team     : {" | ".join(mtg.our_attendees)}',        f'Their Team   : {" | ".join(mtg.their_attendees)}',        f'Deal Stage   : {acct.deal_stage} | Value: {acct.deal_value}',    ]    if acct.health_score > 0:        header_lines.append(f'Health Score  : {acct.health_score}/100')    if acct.contract_end != 'N/A':        header_lines.append(f'Contract End  : {acct.contract_end}')    header = '\n'.join(header_lines)    # ── Executive summary ──    summary_parts = []    summary_parts.append(        f'{co.name} is a {co.industry} company ({co.employee_count} employees, '        f'{co.annual_revenue} revenue) based in {co.hq_location}.'    )    if acct.deal_stage in ('renewal', 'at_risk'):        summary_parts.append(            f'This is a {acct.deal_stage.upper()} meeting. '            f'Deal value: {acct.deal_value}. Sentiment trend: {ctx.sentiment_trend}.'        )    elif acct.deal_stage == 'discovery':        summary_parts.append(            f'This is a DISCOVERY call. Estimated deal: {acct.deal_value}. '            f'Goal: qualify the opportunity and identify pain points.'        )    elif acct.deal_stage == 'expansion':        summary_parts.append(            f'This is an EXPANSION opportunity. Current deal: {acct.deal_value}. '            f'Health score: {acct.health_score}/100.'        )    else:        summary_parts.append(            f'Deal stage: {acct.deal_stage}. Value: {acct.deal_value}.'        )    if mtg.special_notes:        summary_parts.append(f'NOTE: {mtg.special_notes}')    executive_summary = ' '.join(summary_parts)    # ── Risks ──    if ctx.risks:        risks_section = '\n'.join(f'  ! {r}' for r in ctx.risks[:6])    else:        risks_section = '  No significant risks identified.'    # ── Opportunities ──    if ctx.opportunities:        opps_section = '\n'.join(f'  + {o}' for o in ctx.opportunities[:6])    else:        opps_section = '  Discovery stage — opportunities to be identified in this meeting.'    # ── Attendee intel ──    intel_lines = []    for name, info in ctx.attendee_map.items():        icon = {'champion': '*', 'decision_maker': '$$',                'influencer': '^', 'blocker': '!', 'end_user': '-'}.get(info['role'], '-')        intel_lines.append(            f'  [{icon}] {name} ({info["title"]}) — Role: {info["role"]}'        )        if info['notes']:            intel_lines.append(f'      Notes: {info["notes"]}')    attendee_intel = '\n'.join(intel_lines) if intel_lines else '  No prior contact intelligence available.'    # ── Talking points ──    tp_lines = []    for i, tp in enumerate(points):        badge = {'must_say': 'MUST', 'should_say': 'SHOULD', 'nice_to_say': 'NICE'}[tp.priority]        line = f'  {i+1}. [{badge}] [{tp.category}] {tp.text}'        if tp.target_attendee:            line += f' -> {tp.target_attendee}'        tp_lines.append(line)    talking_points_section = '\n'.join(tp_lines)    # ── Prep checklist ──    checklist = []    checklist.append('Review the account timeline (most recent notes above).')    if acct.open_issues:        checklist.append(f'Prepare updates on: {" | ".join(acct.open_issues[:3])}')    if acct.product_usage:        checklist.append('Pull latest usage dashboard for the call.')    for name, info in ctx.attendee_map.items():        if info['role'] == 'decision_maker':            checklist.append(f'Research {name} on LinkedIn (new contact or key decision maker).')    if mtg.meeting_type == 'demo':        checklist.append('Ensure demo environment is running and loaded with relevant sample data.')    if mtg.meeting_type == 'executive':        checklist.append('Prepare executive-level slide deck (max 5 slides).')    checklist.append(f'Block {mtg.duration_min + 15} min (meeting + 15 min post-call debrief).')    prep_checklist = '\n'.join(f'  [ ] {item}' for item in checklist)    brief = CallBrief(        account_id=acct.account_id,        company_name=co.name,        meeting_type=mtg.meeting_type,        urgency=ctx.urgency_level,        header=header,        executive_summary=executive_summary,        risks_section=risks_section,        opportunities_section=opps_section,        attendee_intel=attendee_intel,        talking_points_section=talking_points_section,        prep_checklist=prep_checklist,    )    brief.word_count = len(brief.full_text().split())    return brief# Generate briefs for all accountsbriefs = []for ctx in assembled:    pts = all_points[ctx.account.account_id]    brief = generate_brief(ctx, pts)    briefs.append(brief)for b in briefs:    print(f'{b.account_id}: {b.company_name} | {b.meeting_type} | '          f'urgency={b.urgency} | {b.word_count} words')

## 6 · Sample Call BriefsBelow are complete call briefs for the renewal and support-escalation accounts —the two highest-urgency meetings.

In [ ]:
# Show the two highest-urgency briefsfor brief in sorted(briefs, key=lambda b: {'critical': 0, 'high': 1, 'normal': 2, 'low': 3}[b.urgency])[:2]:    print('=' * 60)    print(brief.full_text())    print('=' * 60)    print()

## 7 · EvaluationWe evaluate the call brief system on four axes:1. **Completeness** — Does the brief cover all required sections?2. **Talking-point coverage** — Are meeting goals reflected in must-say points?3. **Risk coverage** — Are open issues and negative notes surfaced?4. **Brevity** — Is the brief under the target word count for the meeting type?

In [ ]:
REQUIRED_SECTIONS = ['header', 'executive_summary', 'risks_section',                     'opportunities_section', 'attendee_intel',                     'talking_points_section', 'prep_checklist']WORD_TARGETS = {'discovery': 400, 'demo': 500, 'qbr': 600,                'renewal': 600, 'executive': 600, 'support': 500}eval_rows = []for brief, ctx in zip(briefs, assembled):    # Completeness    sections_present = sum(        1 for s in REQUIRED_SECTIONS        if getattr(brief, s, '').strip()    )    completeness = sections_present / len(REQUIRED_SECTIONS)    # Talking-point goal coverage    pts = all_points[brief.account_id]    goal_points = [p for p in pts if p.source == 'meeting_goals']    goal_coverage = len(goal_points) / max(len(ctx.meeting.goals), 1)    # Risk coverage    risk_points = [p for p in pts if p.category in ('risk_mitigation', 'issue_resolution')]    total_risks = len(ctx.risks)    risk_coverage = len(risk_points) / max(total_risks, 1)    # Brevity    target = WORD_TARGETS.get(brief.meeting_type, 500)    brevity_score = min(1.0, target / max(brief.word_count, 1))    eval_rows.append({        'account': brief.company_name,        'type': brief.meeting_type,        'urgency': brief.urgency,        'completeness': round(completeness, 2),        'goal_coverage': round(goal_coverage, 2),        'risk_coverage': round(min(risk_coverage, 1.0), 2),        'brevity': round(brevity_score, 2),        'word_count': brief.word_count,        'talking_points': len(pts),    })eval_df = pd.DataFrame(eval_rows)print('Brief Evaluation Results:')print(eval_df.to_string(index=False))print(f'\nAverage completeness : {eval_df["completeness"].mean():.0%}')print(f'Average goal coverage: {eval_df["goal_coverage"].mean():.0%}')print(f'Average risk coverage: {eval_df["risk_coverage"].mean():.0%}')

## 8 · Visualisations

In [ ]:
# Viz 1: Talking-point priority distribution by accounttp_data = []for acct_id, pts in all_points.items():    brief = next(b for b in briefs if b.account_id == acct_id)    for p in pts:        tp_data.append({            'Account': brief.company_name,            'Priority': p.priority.replace('_', ' ').title(),            'Category': p.category.replace('_', ' ').title(),            'Meeting Type': brief.meeting_type,        })tp_df = pd.DataFrame(tp_data)fig1 = px.histogram(    tp_df, x='Account', color='Priority',    color_discrete_map={        'Must Say': '#e74c3c', 'Should Say': '#f39c12', 'Nice To Say': '#2ecc71'    },    barmode='stack',    title='Talking-Point Priority Distribution by Account',)fig1.update_layout(height=420, xaxis_tickangle=-20)fig1.show()

In [ ]:
# Viz 2: Talking-point category sunburstfig2 = px.sunburst(    tp_df, path=['Meeting Type', 'Category', 'Priority'],    title='Talking Points: Meeting Type > Category > Priority',    color='Priority',    color_discrete_map={        'Must Say': '#e74c3c', 'Should Say': '#f39c12', 'Nice To Say': '#2ecc71'    },)fig2.update_layout(height=550)fig2.show()

In [ ]:
# Viz 3: Evaluation radar per accountcategories = ['Completeness', 'Goal Coverage', 'Risk Coverage', 'Brevity']metric_keys = ['completeness', 'goal_coverage', 'risk_coverage', 'brevity']fig3 = go.Figure()for _, row in eval_df.iterrows():    values = [row[k] for k in metric_keys]    values.append(values[0])    fig3.add_trace(go.Scatterpolar(        r=values,        theta=categories + [categories[0]],        name=f'{row["account"]} ({row["type"]})',        fill='toself', opacity=0.4,    ))fig3.update_layout(    polar=dict(radialaxis=dict(visible=True, range=[0, 1.1])),    title='Brief Quality Radar by Account',    height=520,)fig3.show()

In [ ]:
# Viz 4: Word count vs target by meeting typeeval_df['target'] = eval_df['type'].map(WORD_TARGETS)fig4 = go.Figure()fig4.add_trace(go.Bar(    x=eval_df['account'], y=eval_df['word_count'],    name='Actual Words', marker_color='#3498db',))fig4.add_trace(go.Bar(    x=eval_df['account'], y=eval_df['target'],    name='Target Words', marker_color='#e74c3c', opacity=0.4,))fig4.update_layout(    title='Brief Word Count vs Target',    barmode='overlay', height=420,    yaxis_title='Words',)fig4.show()

In [ ]:
# Viz 5: Risk vs opportunity count by accountbalance_data = []for ctx in assembled:    balance_data.append({        'Account': ctx.company.name,        'Risks': len(ctx.risks),        'Opportunities': len(ctx.opportunities),        'Urgency': ctx.urgency_level,    })bal_df = pd.DataFrame(balance_data)fig5 = go.Figure()fig5.add_trace(go.Bar(    x=bal_df['Account'], y=bal_df['Risks'],    name='Risks', marker_color='#e74c3c',))fig5.add_trace(go.Bar(    x=bal_df['Account'], y=bal_df['Opportunities'],    name='Opportunities', marker_color='#2ecc71',))fig5.update_layout(    title='Risk vs Opportunity Signal Count by Account',    barmode='group', height=420,    xaxis_tickangle=-20,)fig5.show()

In [ ]:
# Viz 6: Health score vs urgency levelmatrix_data = []for ctx, brief in zip(assembled, briefs):    matrix_data.append({        'Account': ctx.company.name,        'Health Score': ctx.account.health_score,        'Urgency': ctx.urgency_level,        'Meeting Type': ctx.meeting.meeting_type,        'Talking Points': len(all_points[ctx.account.account_id]),    })mat_df = pd.DataFrame(matrix_data)fig6 = px.scatter(    mat_df, x='Health Score', y='Urgency', size='Talking Points',    color='Meeting Type', text='Account', size_max=35,    title='Account Health vs Meeting Urgency',)fig6.update_traces(textposition='top center', textfont_size=9)fig6.update_layout(height=450)fig6.show()

In [ ]:
# Viz 7: Sentiment trend across all account notessent_data = []for ctx in assembled:    for note in ctx.account.notes:        sent_data.append({            'Account': ctx.company.name,            'Date': note.date,            'Sentiment': note.sentiment,            'Type': note.note_type,            'Content': note.content[:60],        })sent_df = pd.DataFrame(sent_data)sent_df['Date'] = pd.to_datetime(sent_df['Date'])sent_map = {'positive': 1, 'neutral': 0, 'concern': -0.5, 'negative': -1}sent_df['Score'] = sent_df['Sentiment'].map(sent_map)fig7 = px.scatter(    sent_df, x='Date', y='Account', color='Sentiment', size=[12]*len(sent_df),    color_discrete_map={        'positive': '#2ecc71', 'neutral': '#3498db',        'concern': '#f39c12', 'negative': '#e74c3c',    },    hover_data=['Content', 'Type'],    title='Account Sentiment Timeline',)fig7.update_layout(height=400)fig7.show()

## 9 · Production Architecture### System Design```┌───────────────┐  ┌──────────────┐  ┌───────────────┐│ CRM           │  │ Enrichment   │  │ Calendar      ││ (Salesforce,  │  │ (Clearbit,   │  │ (Google Cal,  ││  HubSpot)     │  │  ZoomInfo)   │  │  Outlook)     │└──────┬────────┘  └──────┬───────┘  └───────┬───────┘       │                  │                   │       └──────────────────┼───────────────────┘                          │                   ┌──────▼───────┐                   │  Context     │                   │  Assembler   │                   └──────┬───────┘                          │              ┌───────────┼───────────┐              │           │           │       ┌──────▼──┐ ┌─────▼────┐ ┌────▼─────┐       │ Talking  │ │ Brief    │ │ Risk     │       │ Point    │ │ Template │ │ Scorer   │       │ Ranker   │ │ Engine   │ │          │       └──────┬───┘ └─────┬────┘ └────┬─────┘              │           │           │              └───────────┼───────────┘                          │                   ┌──────▼───────┐                   │  Call Brief  │                   │  (PDF/Slack/ │                   │   Email)     │                   └──────────────┘```### Key Components| Component | Technology | Purpose ||---|---|---|| CRM connector | Salesforce/HubSpot API | Account history, contacts, deal stage || Enrichment | Clearbit, ZoomInfo, LinkedIn | Company profile, news, tech stack || Calendar | Google Calendar / Outlook | Meeting details, attendees, agenda || Context assembler | Python service | Merge all sources, extract signals || Brief generator | LLM (GPT-4o / Claude) | Natural-language brief from structured context || Talking-point ranker | Rule engine + LLM | Prioritise by meeting type and attendee roles || Delivery | Slack, email, PDF | Push brief 30 min before the meeting |

### Production Code: LangChain Implementation

In [ ]:
PRODUCTION_CODE = r'''# ── call_brief_service.py ─────────────────────────────────────────────from langchain_openai import ChatOpenAIfrom langchain.prompts import ChatPromptTemplatefrom pydantic import BaseModel, Fieldclass TalkingPointOut(BaseModel):    category: str = Field(description="Category: strategic_alignment, risk_mitigation, etc.")    text: str = Field(description="The talking point text")    priority: str = Field(description="must_say, should_say, or nice_to_say")    target_attendee: str = Field(default="", description="Specific person to direct this to")class CallBriefOut(BaseModel):    executive_summary: str = Field(description="2-3 sentence overview")    key_risks: list[str] = Field(description="Top risks for this meeting")    key_opportunities: list[str] = Field(description="Top opportunities")    talking_points: list[TalkingPointOut] = Field(description="Ranked talking points")    prep_checklist: list[str] = Field(description="Preparation checklist items")class CallBriefService:    def __init__(self):        self.llm = ChatOpenAI(model="gpt-4o", temperature=0)        self.prompt = ChatPromptTemplate.from_messages([            ("system", """You are a sales preparation assistant generating call briefs.            Rules:            1. Prioritise talking points by meeting type (renewal = value + risk,               discovery = pain points + qualification, executive = strategic alignment).            2. Always surface open issues as must-say talking points.            3. Include attendee-specific points for decision makers.            4. Keep executive summary under 3 sentences.            5. Preparation checklist should be actionable before the meeting."""),            ("human", """Generate a call brief for this meeting:            Company: {company_context}            Account History: {account_timeline}            Meeting: {meeting_context}            Attendee Intel: {attendee_intel}"""),        ])    def generate(self, company_ctx: str, account_timeline: str,                 meeting_ctx: str, attendee_intel: str) -> CallBriefOut:        chain = self.prompt | self.llm.with_structured_output(CallBriefOut)        return chain.invoke({            "company_context": company_ctx,            "account_timeline": account_timeline,            "meeting_context": meeting_ctx,            "attendee_intel": attendee_intel,        })'''print(PRODUCTION_CODE)

## Summary### What We Built| Component | Description | Key Metric ||---|---|---|| Company profiles | 6 accounts with firmographics, news, tech stack | 6 industries || Account histories | CRM notes, contacts, deal stages, product usage | 5+ notes per account || Meeting contexts | 6 meeting types with attendees, agendas, goals | 6 meeting types || Context assembler | Risk/opportunity extraction, urgency scoring, sentiment trends | Automated signal detection || Talking-point generator | Category-based, priority-ranked, attendee-targeted | Must/Should/Nice ranking || Call brief generator | Structured one-page brief with all sections | 7 required sections || Evaluation | Completeness, goal coverage, risk coverage, brevity | 4-axis quality scoring |### Meeting-Type Adaptations| Type | Primary Focus | Tone ||---|---|---|| Discovery | Qualification, pain points | Curious, consultative || Demo | Technical proof, differentiation | Confident, educational || QBR | ROI, usage trends, expansion | Data-driven, partnership || Renewal | Value, pricing, risk mitigation | Reassuring, evidence-based || Executive | Strategy, outcomes, vision | Concise, high-level || Support | Issue resolution, trust | Empathetic, solution-focused |### Production Path- Connect to CRM (Salesforce/HubSpot) for live account data- Add enrichment APIs (Clearbit, ZoomInfo) for company intelligence- Use an LLM to generate natural-language briefs from the structured context- Deliver via Slack or email 30 minutes before each meeting- Add feedback loop: reps rate brief usefulness to improve generation